# Workshop 3 — Task 3: SVM & Model Evaluation
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Data prepared:', X_train_s.shape)

## 3.1 SVM with Different Kernels

In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
kernel_results = {}

for k in kernels:
    svm = SVC(kernel=k, random_state=42)
    svm.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test_s))
    kernel_results[k] = acc
    print(f'SVM ({k:8s}): {acc:.4f}')

plt.figure(figsize=(8, 4))
plt.bar(kernel_results.keys(), kernel_results.values(),
        color=['#0EA5E9', '#6366F1', '#10B981', '#F59E0B'])
plt.title('SVM Kernel Comparison')
plt.ylabel('Accuracy')
plt.ylim(0.85, 1.0)
plt.tight_layout()
plt.show()

## 3.2 Hyperparameter Tuning (Grid Search)

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.001, 0.01]}
grid = GridSearchCV(SVC(kernel='rbf', random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_s, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV score: {grid.best_score_:.4f}')

best_svm = grid.best_estimator_
y_pred = best_svm.predict(X_test_s)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.4f}')

## 3.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.title('Confusion Matrix — Best SVM')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 3.4 Learning Curve

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_svm, X, y, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#0EA5E9', label='Train', linewidth=2)
plt.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                 train_scores.mean(1)+train_scores.std(1), alpha=0.1, color='#0EA5E9')
plt.plot(train_sizes, val_scores.mean(axis=1), 's-', color='#6366F1', label='Validation', linewidth=2)
plt.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1),
                 val_scores.mean(1)+val_scores.std(1), alpha=0.1, color='#6366F1')
plt.title('Learning Curve — Best SVM')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()